# XGBoost 再学習ノートブック（Stage 3 完了後）

Stage 3 で取得した新データを使ってXGBモデルを再学習する。

**実行順序:** セル1 → セル2 → セル3 → セル4 → セル5 → セル6（検証）

**所要時間:** 約15〜30分（build_training_dataが最長）

In [ ]:
# ── セル1: セットアップ ───────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = '/content/drive/MyDrive/keiba_ai'
sys.path.insert(0, BASE_DIR)
print(f'✅ BASE_DIR: {BASE_DIR}')

In [ ]:
# ── セル2: src/ 強制アップデート ──────────────────────────────
import urllib.request, time as _time
BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
_cb = int(_time.time())  # キャッシュバスター（GitHub CDN対策）
files = [
    'src/tools/__init__.py', 'src/tools/tune_weights.py',
    'src/tools/calibrate.py', 'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py', 'src/tools/build_training_data.py',
    'src/tools/train_xgb.py', 'src/tools/calibrate_xgb.py',
    'src/features/engine.py',
    'src/utils/config.py', 'src/utils/db.py', 'src/utils/model_registry.py',
    'src/scraper/parser.py', 'src/scraper/jra_scraper.py',
    'src/models/__init__.py', 'src/models/calibration.py',
    'src/models/calibration_xgb.py', 'src/models/predict.py',
    'src/betting/__init__.py', 'src/betting/make_bets.py',
    'src/betting/ev_filter.py', 'src/betting/app_json.py',
]
for rel in files:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}?nocache={_cb}', dest)
    print(f'OK {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

# ── 新特徴量が含まれているか確認 ──
with open(f'{BASE_DIR}/src/features/engine.py') as _f:
    _eng_src = _f.read()
for _kw in ['f_sex', 'f_age', 'f_track_cond', 'f_class_level']:
    _ok = _kw in _eng_src
    print(f'  {"✅" if _ok else "❌"} engine.py に {_kw} {"あり" if _ok else "なし！← 要確認"}')

print('done')

In [ ]:
# ── セル3: 学習データ生成（horse_features.csv）────────────────
# 約10〜20分かかる
# 既存CSVは horse_features_old.csv に自動バックアップされる

from src.tools.build_training_data import build_training_data
df = build_training_data(BASE_DIR)

print(f'\n── 新特徴量確認 ──')
new_cols = ['f_sex', 'f_age', 'f_track_cond', 'f_heavy_track_rate',
            'f_class_level', 'f_class_jump', 'f_finish_time_avg', 'f_time_diff_avg']
for c in new_cols:
    if c in df.columns:
        non_null = df[c].notna().sum()
        pct = 100 * non_null / len(df)
        print(f'  ✅ {c}: {pct:.1f}% 充足')
    else:
        print(f'  ❌ {c}: 列なし')

In [ ]:
# ── セル4: XGBoost 再学習 ─────────────────────────────────────
# 旧モデルは xgb_fukusho_model_old.pkl に自動退避
# AUC が旧モデルを下回る場合は正式採用されない（手動確認）

from src.tools.train_xgb import train_xgb

result = train_xgb(
    BASE_DIR,
    train_end='2026-03-31',
    val_start='2026-04-01',
    val_end='2026-05-31',
    n_estimators=500,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=10,
    reg_alpha=0.1,
    reg_lambda=1.0,
    early_stopping_rounds=50,
)

print(f'\n── 合格基準チェック ──')
auc_ok    = result['auc']    >= 0.77
brier_ok  = result['brier']  <= 0.150
n_feat_ok = result['n_features'] >= 65
print(f'  AUC   {result["auc"]:.4f}  {"✅" if auc_ok    else "⚠"} (≥0.77)')
print(f'  Brier {result["brier"]:.4f}  {"✅" if brier_ok  else "⚠"} (≤0.150)')
print(f'  特徴量 {result["n_features"]}   {"✅" if n_feat_ok else "⚠"} (≥65)')

In [ ]:
# ── セル5: XGBキャリブレーター再学習 ─────────────────────────
# 新モデルの raw_prob 分布が変わるため再学習が必要

for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]

from src.tools.calibrate_xgb import run_xgb_calibration
run_xgb_calibration(BASE_DIR, test_days=60, n_bins=20)

In [ ]:
# ── セル6: 統合テスト ──────────────────────────────────────────
import pickle, json
import numpy as np

model_path    = f'{BASE_DIR}/data/xgb_fukusho_model.pkl'
cols_path     = f'{BASE_DIR}/data/xgb_feature_cols.json'
calib_path    = f'{BASE_DIR}/data/xgb_calibrator.pkl'

with open(model_path, 'rb') as f:  model = pickle.load(f)
with open(cols_path)         as f:  cols  = json.load(f)['feature_cols']
with open(calib_path, 'rb')  as f:  calib = pickle.load(f)

print(f'✅ モデル: {model_path}')
print(f'✅ 特徴量数: {len(cols)}')
print(f'✅ キャリブレーター: {type(calib).__name__}')

# テスト: 16頭立てダミーデータで確率合計 ≒ 3.0
import pandas as pd
dummy = pd.DataFrame([{c: np.random.normal(5.0, 1.5) for c in cols} for _ in range(16)])
raw_probs = model.predict_proba(dummy)[:, 1]

# IsotonicCalibrator は .transform() メソッドを使う
cal_probs = np.array(calib.transform(raw_probs))

print(f'\n── ダミー16頭 ──')
print(f'  raw_prob range: {raw_probs.min():.3f} ~ {raw_probs.max():.3f}')
print(f'  cal_prob 合計: {cal_probs.sum():.3f}  (2.8〜3.2 が正常)')
print(f'  cal_prob 分散: {raw_probs.max() - raw_probs.min():.3f}  (>0.3 が正常)')

calib_ok  = 2.8 <= cal_probs.sum() <= 3.2
spread_ok = raw_probs.max() - raw_probs.min() > 0.3
print(f'  cal_prob合計 {"✅" if calib_ok  else "⚠"}')
print(f'  分散 {"✅" if spread_ok else "⚠"}')

# 特徴量重要度トップ20
imps = sorted(zip(cols, model.feature_importances_), key=lambda x: x[1], reverse=True)[:20]
print('\n── 特徴量重要度 Top 20 ──')
for name, imp in imps:
    marker = '★' if name in ('f_cl_rank', 'f_rl_rank', 'f_weight_load',
                               'f_sex', 'f_age', 'f_track_cond',
                               'f_class_level', 'f_time_diff_avg') else ''
    print(f'  {name:<40} {imp*100:.2f}% {marker}')